# Obese liver after a glucose load: a genome-wide time course

**Question.** How does the liver's response to glucose differ between lean
and obese mice, genome-wide and over time, and which regulatory mechanisms
carry each response?

**Data.** Wild-type and leptin-deficient obese (ob/ob) mouse liver, sampled
at 0, 20, 60, 120 and 240 minutes after an oral glucose load. The
transcriptome covers 14,292 genes and the metabolome 162 compounds, each with
the authors' own fold changes, q-values and half-response times. The
measurements were published by Kokaji *et al.*, *Science Signaling*
13(660):eaaz1236, 2020, and are downloaded when the notebook runs.

The panel study (`obese_liver.py`) looks at the same comparison on 19 genes at
one time point; this notebook covers the whole liver over four hours. One
analysis appears only here: measured enzyme affinities show whether a
metabolite's change can matter to its enzyme at all.

In [1]:
import json
import warnings

import matplotlib.pyplot as plt
import pandas as pd

from transnet import (
    compare_transomic_networks,
    convergence_significance,
    cross_layer_connectivity,
    downstream_influence,
    layer_coverage,
    map_omics_to_network,
    metabolite_regulatory_roles,
    reaction_regulation_table,
    regulation_axis_summary,
    regulatory_motifs,
    regulatory_role_enrichment,
    responsive_subnetwork,
    structural_vulnerability,
    trace_regulatory_paths,
    transcription_factor_activity,
)
from transnet.analysis import hub_rankings, path_verdicts, versus_chance
from transnet.datasets import (
    DATA_DIR,
    KOKAJI_TIMEPOINTS,
    fetch_kokaji_tables,
    kokaji_contrast,
    load_kokaji_table,
)
from transnet.io import read_network

warnings.filterwarnings("ignore", category=RuntimeWarning)

OUT = DATA_DIR / "published_results" / "liver_timecourse"
OUT.mkdir(parents=True, exist_ok=True)
TIMEPOINT = 240
LOG2FC_THRESHOLD = 0.585        # 1.5-fold, as the Kuroda laboratory defines it
QVALUE_THRESHOLD = 0.1

fetch_kokaji_tables()
source = DATA_DIR / "mouse" / "latest"
network = read_network(str(source / "interactions.csv"),
                       nodes_file=str(source / "nodes.csv"))
print(f"{network.number_of_nodes():,} molecules, "
      f"{network.number_of_edges():,} relationships")

106,859 molecules, 1,938,789 relationships


In [2]:
from transnet.visualization import plot_layer_connectivity

connectivity = cross_layer_connectivity(network)
print(f"{connectivity['cross_layer_fraction']:.0%} of edges cross between layers")
layer_figure = plot_layer_connectivity(connectivity,
                                       title="Mouse network: edges between layers")
plt.show()

82% of edges cross between layers


/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/1619897945.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## The contrasts

No statistics are recomputed. The supplement gives a fold change, p-value
and q-value per time point and genotype, so each contrast is read from it and
the ratio converted to log2. The main contrast is 240 minutes against 0
minutes.

The metabolites come with KEGG compound ids, which the network also uses, so
no name matching is needed.

In [3]:
contrasts = {}
for genotype in ("WT", "ob/ob"):
    contrasts[genotype] = {
        "Metabolome": kokaji_contrast("S1", genotype, TIMEPOINT),
        "Transcriptome": kokaji_contrast("S4", genotype, TIMEPOINT),
    }

pd.DataFrame([
    {"genotype": genotype, "layer": layer, "features": len(table),
     "q <= 0.1": int((table["padj"] <= QVALUE_THRESHOLD).sum()),
     "and 1.5-fold": int(((table["padj"] <= QVALUE_THRESHOLD)
                          & (table["log2FC"].abs() >= LOG2FC_THRESHOLD)).sum())}
    for genotype, tables in contrasts.items() for layer, table in tables.items()
]).set_index(["genotype", "layer"])

features  q <= 0.1  and 1.5-fold
genotype layer                                          
WT       Metabolome          162        36            26
         Transcriptome     14292       425           284
ob/ob    Metabolome          162        11             8
         Transcriptome     14292       552           399

The first result is visible before any network is used: at 240 minutes more
metabolites change in wild-type than in obese liver, and more genes change in
obese than in wild-type liver.

## The response over time

The same comparison at every time point. Each column is one genotype at one
time point; each row is a metabolite that changed at some time point. Rows are
sorted by their largest change.

In [4]:
from transnet.visualization import plot_similarity_heatmap, plot_values_heatmap

over_time, changed_sets = {}, {}
for genotype in ("WT", "ob/ob"):
    for minutes in KOKAJI_TIMEPOINTS:
        column = f"{genotype} {minutes} min"
        for layer, sheet in (("Metabolome", "S1"), ("Transcriptome", "S4")):
            contrast = kokaji_contrast(sheet, genotype, minutes)
            hit = contrast[(contrast["padj"] <= QVALUE_THRESHOLD)
                           & (contrast["log2FC"].abs() >= LOG2FC_THRESHOLD)]
            changed_sets.setdefault(layer, {})[column] = set(hit["feature"])
            if layer == "Metabolome":
                over_time[column] = contrast.set_index("feature")["log2FC"]

values = pd.DataFrame(over_time)
values = values.loc[values.index.isin(set().union(*changed_sets["Metabolome"].values()))]
timecourse_figure = plot_values_heatmap(values, graph=network, max_rows=35,
                                        title="Changed metabolites, by genotype and time")
plt.show()

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/2708374977.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


How similar are the responses? The Jaccard index of two sets of changed genes
is the number they share divided by the number in either: 1 means the same
genes changed, 0 means none in common.

In [5]:
columns = list(changed_sets["Transcriptome"])
jaccard = pd.DataFrame(
    [[len(changed_sets["Transcriptome"][a] & changed_sets["Transcriptome"][b])
      / max(len(changed_sets["Transcriptome"][a] | changed_sets["Transcriptome"][b]), 1)
      for b in columns] for a in columns],
    index=columns, columns=columns)
similarity_figure = plot_similarity_heatmap(
    jaccard, title="Overlap of the changed genes between time points and genotypes")
plt.show()

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/2740462218.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Identifiers

Genes are identified by Ensembl ids and the network uses Entrez ids.
Translating them takes thousands of lookups, so the result is cached.

In [6]:
def cached_map(name, features, from_type, to_type):
    path = OUT / f"idmap_{name}.json"
    if path.exists():
        return json.loads(path.read_text())
    from transnet.analysis import id_mapping
    mapped = id_mapping(pd.DataFrame({"feature": sorted(features)}), id_col="feature",
                        from_type=from_type, to_type=to_type, organism="mouse")
    column = f"{to_type}_id"
    result = {k: str(v) for k, v in zip(mapped["feature"], mapped[column])
              if pd.notna(v)}
    path.write_text(json.dumps(result))
    return result


genes = set(contrasts["WT"]["Transcriptome"]["feature"])
metabolite_ids = set(contrasts["WT"]["Metabolome"]["feature"])
id_map = {
    "Transcriptome": cached_map("ensembl_to_entrez", genes, "ensembl.gene", "entrezgene"),
    "Metabolome": {i: i for i in metabolite_ids},      # already KEGG compounds
}
{layer: len(m) for layer, m in id_map.items()}

{'Transcriptome': 13413, 'Metabolome': 162}

## A network per genotype

In [7]:
graphs, reports = {}, []
for genotype, tables in contrasts.items():
    graph = network.copy()
    report = map_omics_to_network(
        graph, tables, id_column="feature", log2fc_column="log2FC",
        qvalue_column="padj", id_map=id_map,
        qvalue_threshold=QVALUE_THRESHOLD, log2fc_threshold=LOG2FC_THRESHOLD,
    )
    graphs[genotype] = graph
    reports.append(report.per_layer.assign(genotype=genotype))

pd.concat(reports)[["genotype", "layer", "n_supplied", "n_matched",
                    "match_fraction", "n_up", "n_down"]].reset_index(drop=True)

,genotype,layer,n_supplied,n_matched,match_fraction,n_up,n_down
0,WT,Metabolome,162,162,1.000000,25,1
1,WT,Transcriptome,14292,14024,0.981248,362,647
2,ob/ob,Metabolome,162,162,1.000000,8,0
3,ob/ob,Transcriptome,14292,14024,0.981248,413,642


In [8]:
layer_coverage(graphs["WT"])

,layer,n_nodes,n_measured,measured_fraction,n_responsive,n_regulated,n_up,n_down,mean_degree
0,Transcriptome,57540,14023,0.243709,1009,1009,362,647,26.297098
1,Proteome,17283,0,0.000000,0,0,0,0,130.570966
2,Reactions,12384,0,0.000000,0,0,0,0,5.680071
3,Metabolome,19652,162,0.008243,26,26,25,1,1.905302


## Which axis regulates each reaction

There is no proteome here, so the enzyme axis rests on transcripts alone.
`gene_axis_evidence` records this (`gene` rather than `gene_protein`), and it
is why these counts cannot be compared directly with a study that measured
proteins.

In [9]:
regulation, rows = {}, []
for genotype, graph in graphs.items():
    table = reaction_regulation_table(graph)
    regulation[genotype] = table
    regulated = table[(table["gene_axis"] != 0) | (table["metabolite_axis"] != 0)]
    rows.append({
        "genotype": genotype,
        "regulated": len(regulated),
        "enzyme only": int(((regulated["gene_axis"] != 0)
                            & (regulated["metabolite_axis"] == 0)).sum()),
        "metabolite only": int(((regulated["gene_axis"] == 0)
                                & (regulated["metabolite_axis"] != 0)).sum()),
        "both": int(((regulated["gene_axis"] != 0)
                     & (regulated["metabolite_axis"] != 0)).sum()),
        "controversial": int(regulated["controversial"].sum()),
    })
    table.to_csv(OUT / f"reaction_regulation_{genotype.replace('/', '')}.csv",
                 index=False)

axes = pd.DataFrame(rows).set_index("genotype")
axes

,regulated,enzyme only,metabolite only,both,controversial
genotype,,,,,
WT,1202,790,304,108,62
ob/ob,844,756,75,13,8


Healthy liver responds to glucose mainly through its metabolites, obese liver
mainly through gene expression. Over the whole liver, the two axes can be
counted in each genotype.

In [10]:
from transnet.visualization import plot_axis_composition

composition = pd.DataFrame([
    {"contrast": genotype,
     "enzyme_axis_only": row["enzyme only"],
     "metabolite_axis_only": row["metabolite only"],
     "both_axes": row["both"],
     "controversial": row["controversial"]}
    for genotype, row in axes.iterrows()
])
plot_axis_composition(composition)
plt.show()

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/2442311944.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### The controversial reactions

Where the two axes point in opposite directions. Each bar is one measured
molecule's push on the reaction: to the right it speeds the reaction up, to
the left it slows it down.

In [11]:
from transnet.visualization import plot_controversial_reactions

controversial_figures = {}
for genotype, graph in graphs.items():
    table = regulation[genotype]
    print(f"{genotype}: {int(table['controversial'].sum())} controversial reactions")
    controversial_figures[genotype] = plot_controversial_reactions(
        graph, table, max_enzymes=5,
        title=f"{genotype} liver: controversial reactions")
    plt.show()

WT: 62 controversial reactions
ob/ob: 8 controversial reactions


/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/3990723244.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/3990723244.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Per-pathway balance, and the metabolites doing the regulating

`kegg_reaction_pathways` assigns each reaction to its mouse KEGG pathways, and
`regulation_axis_summary` counts, per pathway, the reactions each axis
activates or inhibits. The figures show the pathways with the most regulated
reactions in each genotype.

In [12]:
from transnet.api import kegg_reaction_pathways
from transnet.visualization import plot_regulation_axes

reactions = [n for n, d in network.nodes(data=True) if d.get("layer") == "Reactions"]
pathways = kegg_reaction_pathways(reactions, organism="mmu")
balance = {genotype: regulation_axis_summary(table, pathway_map=pathways)
           for genotype, table in regulation.items()}
pd.concat([b.assign(genotype=g) for g, b in balance.items()]).set_index("genotype").head(12)

,pathway,n_reactions,gene_activated,gene_inhibited,metabolite_activated,metabolite_inhibited,n_controversial,fraction_controversial,fraction_gene_driven,fraction_metabolite_driven
genotype,,,,,,,,,,
WT,unassigned,623,192,316,70,86,20,0.032103,0.815409,0.250401
WT,Purine metabolism,39,8,6,6,23,3,0.076923,0.358974,0.743590
WT,Glycerophospholipid metabolism,29,4,14,14,4,6,0.206897,0.620690,0.620690
WT,Pyrimidine metabolism,24,6,5,4,14,1,0.041667,0.458333,0.750000
WT,Biosynthesis of unsaturated fatty acids,23,0,23,11,0,11,0.478261,1.000000,0.478261
WT,Metabolism of xenobiotics by cytochrome P450,22,19,0,0,3,0,0.000000,0.863636,0.136364
WT,Biosynthesis of various nucleotide sugars,18,0,10,6,4,2,0.111111,0.555556,0.555556
WT,Fatty acid degradation,17,0,17,0,0,0,0.000000,1.000000,0.000000
WT,Sphingolipid metabolism,16,4,7,4,1,0,0.000000,0.687500,0.312500


In [13]:
axis_figures = {}
for genotype, summary in balance.items():
    top = summary[summary["pathway"] != "unassigned"].head(12)
    axis_figures[genotype] = plot_regulation_axes(
        top, title=f"{genotype} liver, 240 min after glucose: regulation by pathway")
    plt.show()

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/3481575196.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/3481575196.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
role_rows = []
for genotype, graph in graphs.items():
    result = regulatory_role_enrichment(metabolite_regulatory_roles(graph))
    counts = result["counts"]
    row = result["enrichment"].set_index("role").loc["any"]
    role_rows.append({
        "genotype": genotype,
        "changed metabolites": counts["n_differential"],
        "of those regulators": counts["n_differential_regulators"],
        "background share": round(counts["fraction_background_regulators"], 3),
        "q": round(float(row["q_value"]), 3),
    })
pd.DataFrame(role_rows).set_index("genotype")

,changed metabolites,of those regulators,background share,q
genotype,,,,
WT,26,14,0.488,0.544
ob/ob,8,3,0.488,0.845


In [15]:
from transnet.visualization import plot_metabolite_regulators

regulator_figure = plot_metabolite_regulators(
    regulatory_role_enrichment(metabolite_regulatory_roles(graphs["WT"])))
plt.show()

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/743690692.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Saturation, and the currency metabolites

A metabolite can be an annotated regulator of an enzyme and still be irrelevant
to it if its concentration sits far from the affinity. Table S13 gives measured
Km and Ki values with a saturation index per genotype, for ATP and NADP+.

Those two are exactly what TransNet treats as *currency* metabolites and
excludes from the metabolite axis, on the grounds that everything uses them and
a change in ATP is not a regulatory statement about any particular reaction.
The saturation index tests that decision rather than assuming it: if these
enzymes are already near-saturated for ATP, then ATP moving changes nothing,
and excluding it is right.

In [16]:
kinetics = load_kokaji_table("S13", sheet="Km, Ki (ATP, NADP+)", header_rows=1)
kinetics = kinetics.assign(
    wt=pd.to_numeric(kinetics["Saturation index of WT mice at 0min"], errors="coerce"),
    obese=pd.to_numeric(kinetics["Saturation index of ob/ob mice at 0min"],
                        errors="coerce"),
).dropna(subset=["wt", "obese"])

print(f"{len(kinetics)} enzyme-metabolite pairs with a measured constant")
kinetics.groupby(["Km or Ki", "Metabolite"])[["wt", "obese"]].mean().round(3)

93 enzyme-metabolite pairs with a measured constant


wt  obese
Km or Ki Metabolite              
Ki       ATP         0.165  0.416
Km       ATP         0.711  0.869
         NADP+       0.825  0.818

A saturation index near 1 means the enzyme is saturated, so a further rise in
that metabolite changes nothing. Comparing the genotypes says whether obesity
moves an enzyme into or out of the range where its regulator still has an
effect.

In [17]:
shifted = kinetics.assign(shift=kinetics["obese"] - kinetics["wt"])
shifted.nlargest(8, "shift")[["Km or Ki", "Metabolite", "EC number",
                              "Geometric mean", "wt", "obese", "shift"]]

,Km or Ki,Metabolite,EC number,Geometric mean,wt,obese,shift
78,Ki,ATP,2.7.7.4,1.048809,0.323341,0.668620,0.345278
12,Km,ATP,2.7.4.8,1,0.333854,0.679092,0.345238
4,Km,ATP,2.7.1.48,1.134938,0.306320,0.650906,0.344587
56,Km,ATP,2.7.6.2,1.2,0.294605,0.638136,0.343531
59,Km,ATP,3.6.1.3,0.88,0.362861,0.706291,0.343430
17,Km,ATP,6.2.1.3,1.4,0.263613,0.601838,0.338226
79,Ki,ATP,2.7.8.11,1.4,0.263613,0.601838,0.338226
14,Km,ATP,2.7.9.3,0.75,0.400563,0.738326,0.337763


Two aggregate means would hide the shape of this. Every pair, with the
diagonal: a point above the line is an enzyme sitting closer to saturation in
obese liver, where a further change in that metabolite has less effect.

In [18]:
from transnet.visualization.palette import DOWN, INK, MUTED, SURFACE, UP, style_axes

saturation_figure, saturation_axes = plt.subplots(figsize=(5.2, 5))
saturation_axes.plot([0, 1], [0, 1], color=MUTED, linewidth=1, zorder=1)
for constant, colour in (("Km", UP), ("Ki", DOWN)):
    subset = shifted[shifted["Km or Ki"] == constant]
    saturation_axes.scatter(subset["wt"], subset["obese"], s=34, alpha=0.75,
                            color=colour, edgecolor=SURFACE, linewidth=0.6,
                            label=f"{constant} (n = {len(subset)})", zorder=2)
above = int((shifted["shift"] > 0).sum())
saturation_axes.set_xlabel("saturation index, wild-type liver")
saturation_axes.set_ylabel("saturation index, obese liver")
saturation_axes.set_title(
    f"ATP and NADP+ saturation\n{above} of {len(shifted)} pairs sit closer to "
    f"saturation in obese liver", loc="left", fontsize=11, fontweight="bold",
    color=INK)
saturation_axes.legend(frameon=False, fontsize=9, loc="lower right")
style_axes(saturation_axes)
saturation_figure.tight_layout()
plt.show()

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/2333719595.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The same pairs, matched to the reactions the network regulated, so the
saturation index attaches to a regulation call rather than to an EC number on
its own.

In [19]:
# The EC number sits on the catalysis edge, not on the reaction node, because one
# reaction can be catalysed by enzymes of more than one EC class.
reaction_ec = {}
for enzyme, reaction, data in graphs["WT"].edges(data=True):
    if data.get("edge_type") != "catalysis":
        continue
    for ec in str(data.get("ec") or "").split(","):
        ec = ec.strip()
        if ec:
            reaction_ec.setdefault(reaction, set()).add(ec)

ec_saturation = (shifted.groupby(shifted["EC number"].astype(str).str.strip())
                 [["wt", "obese", "shift"]].mean().round(3))

rows = []
for reaction, ecs in reaction_ec.items():
    matched = [ec for ec in ecs if ec in ec_saturation.index]
    if not matched:
        continue
    row = regulation["WT"].loc[regulation["WT"]["reaction"] == reaction]
    if row.empty:
        continue
    row = row.iloc[0]
    if row["gene_axis"] == 0 and row["metabolite_axis"] == 0:
        continue
    values = ec_saturation.loc[matched].mean()
    rows.append({"reaction": reaction, "name": str(row["name"])[:44],
                 "ec": ", ".join(sorted(matched)),
                 "gene_axis": row["gene_axis"],
                 "metabolite_axis": row["metabolite_axis"],
                 "wt": round(values["wt"], 3), "obese": round(values["obese"], 3),
                 "shift": round(values["shift"], 3)})

regulated_with_kinetics = pd.DataFrame(rows)
print(f"{len(regulated_with_kinetics)} regulated reactions have a measured "
      f"saturation index")
regulated_with_kinetics.sort_values("shift", key=abs, ascending=False).head(10)

87 regulated reactions have a measured saturation index


,reaction,name,ec,gene_axis,metabolite_axis,wt,obese,shift
85,R08232,ATP:5-fluorouridine 5'-phosphotransferase; 5,2.7.1.48,0,-1,0.306,0.651,0.345
84,R00968,GTP:uridine 5'-phosphotransferase; GTP + Uri,2.7.1.48,0,-1,0.306,0.651,0.345
83,R00964,ATP:uridine 5'-phosphotransferase; ATP + Uri,2.7.1.48,0,-1,0.306,0.651,0.345
82,R00517,GTP:cytidine 5'-phosphotransferase; GTP + Cy,2.7.1.48,0,-1,0.306,0.651,0.345
81,R00513,ATP:cytidine 5'-phosphotransferase; ATP + Cy,2.7.1.48,0,-1,0.306,0.651,0.345
68,R12852,guanylate kinase; dZMP + ATP <=> dZDP + ADP,2.7.4.8,0,-1,0.334,0.679,0.345
67,R02090,ATP:dGMP phosphotransferase; ATP + dGMP <=>,2.7.4.8,0,-1,0.334,0.679,0.345
66,R00332,ATP:GMP phosphotransferase; ATP + GMP <=> AD,2.7.4.8,0,-1,0.334,0.679,0.345
0,R00390,long-chain-fatty-acid:CoA ligase (AMP-formin,6.2.1.3,-1,0,0.264,0.602,0.338
1,R01280,palmitate:CoA ligase (AMP-forming); ATP + He,6.2.1.3,-1,0,0.264,0.602,0.338


## Transcription factors

`transcription_factor_activity` asks which factors' ChIP-Atlas target genes
are over-represented among the genes that changed.

In [20]:
activity = {}
for genotype, graph in graphs.items():
    table = transcription_factor_activity(graph, min_targets=5)
    activity[genotype] = table
    implicated = table[table["q_value"] <= 0.05] if not table.empty else table
    print(f"{genotype:<6} {len(implicated)} of {len(table)} factors implicated "
          f"at q <= 0.05")

activity["WT"].head(12)[["name", "n_targets", "n_responsive_targets", "n_up",
                         "n_down", "q_value", "inferred_activity",
                         "factor_regulated"]]

WT     10 of 708 factors implicated at q <= 0.05


ob/ob  9 of 708 factors implicated at q <= 0.05


,name,n_targets,n_responsive_targets,n_up,n_down,q_value,inferred_activity,factor_regulated
0,Jarid2,1101,172,69,103,7.389843e-21,0,None
1,Suz12,930,146,58,88,9.986667e-18,0,None
2,Cbx7,851,136,55,81,3.239382e-17,0,None
3,Rnf2,1764,216,92,124,4.136176e-14,0,None
4,Mtf2,642,100,45,55,1.109390e-11,0,None
5,Eed,440,75,31,44,1.651585e-10,0,None
6,Pcgf2,792,111,45,66,3.935012e-10,0,None
7,Ezh2,249,41,16,25,4.235599e-05,0,None
8,Bmi1,76,16,8,8,6.424744e-03,0,None
9,Phc1,179,27,13,14,1.371096e-02,0,None


In [21]:
from transnet.visualization import plot_tf_activity

tf_figure = plot_tf_activity(activity["WT"],
                             title="Transcription factors, wild-type liver")
plt.show()

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/1353390855.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Response timing

The other studies compute half-response times with TransNet. This supplement
provides the authors' own, per genotype and for both layers, so the question
of whether well-connected molecules respond first can be asked with those
values.

In [24]:
timing = []
for genotype in ("WT", "ob/ob"):
    for layer, table in (("Metabolome", "S1"), ("Transcriptome", "S4")):
        contrast = kokaji_contrast(table, genotype, TIMEPOINT)
        if "t_half" not in contrast:
            continue
        timed = contrast.dropna(subset=["t_half"])
        timing.append({"genotype": genotype, "layer": layer,
                       "with a t-half": len(timed),
                       "median t-half (min)": round(timed["t_half"].median(), 1)})
pd.DataFrame(timing).set_index(["genotype", "layer"])

with a t-half  median t-half (min)
genotype layer                                            
WT       Metabolome                64                 13.2
         Transcriptome           1255                 26.8
ob/ob    Metabolome                46                 14.7
         Transcriptome           2511                 15.9

Morita *et al.* report that in healthy liver the best-connected molecules
respond first, and that this relationship is lost in obesity. The same test,
on the authors' half-response times and this network's connections:

In [25]:
from scipy import stats

degree_timing = []
for genotype, graph in graphs.items():
    for layer, table in (("Metabolome", "S1"), ("Transcriptome", "S4")):
        contrast = kokaji_contrast(table, genotype, TIMEPOINT)
        if "t_half" not in contrast:
            continue
        mapped = contrast.assign(
            node=contrast["feature"].map(lambda f: id_map[layer].get(f, f)))
        mapped = mapped[mapped["node"].isin(graph)].dropna(subset=["t_half"])
        if len(mapped) < 10:
            continue
        degrees = mapped["node"].map(dict(graph.degree()))
        rho, p = stats.spearmanr(degrees, mapped["t_half"])
        degree_timing.append({"genotype": genotype, "layer": layer,
                              "n": len(mapped), "rho": round(rho, 3),
                              "p": float(f"{p:.3g}")})
pd.DataFrame(degree_timing).set_index(["genotype", "layer"])

n    rho       p
genotype layer                             
WT       Metabolome       64 -0.000  0.9990
         Transcriptome  1238 -0.018  0.5340
ob/ob    Metabolome       46 -0.156  0.3020
         Transcriptome  2484 -0.034  0.0919

`temporal_network_structure` runs the same test once the half-response times
are stored on the network's nodes, and `plot_temporal_structure` draws it.

In [26]:
from transnet import temporal_network_structure
from transnet.visualization import plot_temporal_structure

timed_graph = graphs["WT"].copy()
for layer, sheet in (("Metabolome", "S1"), ("Transcriptome", "S4")):
    contrast = kokaji_contrast(sheet, "WT", TIMEPOINT)
    if "t_half" not in contrast:
        continue
    for feature, t_half in zip(contrast["feature"], contrast["t_half"]):
        node = id_map[layer].get(feature, feature)
        if node in timed_graph and pd.notna(t_half):
            timed_graph.nodes[node]["t_half"] = float(t_half)
timing_structure = temporal_network_structure(timed_graph)
print(timing_structure["degree_vs_thalf"]["interpretation"])
timing_figure = plot_temporal_structure(timed_graph, timing_structure, time_unit="min",
                                        title="Wild-type liver: connections against timing")
plt.show()

no association between degree and response time (p = 0.627, n = 1302)


/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/3414493901.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Hubs

In [27]:
hub_tables = {}
for genotype, graph in graphs.items():
    hubs, without_binding, binding_share = hub_rankings(graph, top_percent=2)
    hub_tables[genotype] = hubs
    print(f"{genotype:<6} " + ", ".join(hubs.head(3)["name"].str[:34]))

hub_tables["WT"].head(10)[["name", "layer", "cross_layer_degree",
                           "n_layers_touched"]]

WT     ATP; Adenosine 5'-triphosphate, GTP; Guanosine 5'-triphosphate, UTP; Uridine 5'-triphosphate; Urid
ob/ob  D-Fructose 6-phosphate; D-Fructose, D-Glucose 6-phosphate; Glucose 6-p


,name,layer,cross_layer_degree,n_layers_touched
0,ATP; Adenosine 5'-triphosphate,Metabolome,196,1
1,GTP; Guanosine 5'-triphosphate,Metabolome,117,1
2,UTP; Uridine 5'-triphosphate; Uridine triphosp...,Metabolome,66,1
3,UDP; Uridine 5'-diphosphate,Metabolome,41,1
4,L-Glutamine; L-2-Aminoglutaramic acid,Metabolome,38,1
5,L-Leucine; 2-Amino-4-methylvaleric acid; (2S)-...,Metabolome,27,1
6,Cytidine,Metabolome,22,1
7,Phosphoenolpyruvate; Phosphoenolpyruvic acid; PEP,Metabolome,18,1
8,dGTP; 2'-Deoxyguanosine 5'-triphosphate; Deoxy...,Metabolome,15,1
9,CDP; Cytidine 5'-diphosphate; Cytidine diphosp...,Metabolome,9,1


In [28]:
from transnet.visualization import plot_transomic_hubs

hub_figure = plot_transomic_hubs(hub_tables["WT"], title="Wild-type liver: molecules "
                                                          "connecting layers")
plt.show()

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/2187035970.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Signed paths and propagation

In [29]:
traced, traced_paths, influences, path_rows = {}, {}, {}, []
for genotype, graph in graphs.items():
    paths = trace_regulatory_paths(graph, target_layer="Metabolome", max_length=4,
                                   max_paths=20000)
    summary, agree_paths, tested_paths = path_verdicts(paths)
    traced[genotype] = (summary, agree_paths, tested_paths)
    traced_paths[genotype] = paths
    seeds = {n: float(d["log2fc"]) for n, d in graph.nodes(data=True)
             if d.get("layer") == "Transcriptome" and d.get("regulated")
             and d.get("log2fc") is not None}
    influence = downstream_influence(graph, seeds, target_layer="Metabolome")
    influences[genotype] = influence
    measured = influence[influence["observed"].fillna(0) != 0]
    agree = int(measured["agrees"].astype("boolean").fillna(False).sum())
    path_rows.append({
        "genotype": genotype,
        "paths": len(paths),
        "metabolites the paths score": tested_paths,
        "paths predict correctly": agree_paths,
        "propagation reaches": len(measured),
        "propagation correct": agree,
    })
pd.DataFrame(path_rows).set_index("genotype")

,paths,metabolites the paths score,paths predict correctly,propagation reaches,propagation correct
genotype,,,,,
WT,5788,0,0,22,8
ob/ob,1388,0,0,6,2


In [30]:
for genotype, (summary, agree_paths, tested_paths) in traced.items():
    if not tested_paths:
        print(f"{genotype}: no path reaches a changed metabolite")
        continue
    print(f"{genotype}: {agree_paths} of {tested_paths} metabolites predicted "
          f"correctly: {versus_chance(agree_paths, tested_paths)}")

WT: no path reaches a changed metabolite
ob/ob: no path reaches a changed metabolite


Thousands of paths reach the metabolome, but only 162 metabolites were
measured, so few paths end at a metabolite whose change can be checked. The
path figure therefore mostly shows predictions without a measurement to test
them against. Propagation gives every measured metabolite a predicted
direction, and `plot_downstream_influence` compares it with the measurement.

In [31]:
from transnet.visualization import plot_downstream_influence, plot_regulatory_paths

paths_figure = plot_regulatory_paths(traced_paths["WT"], graphs["WT"], top_n=8)
plt.show()

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/3561295043.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [32]:
influence_figure = plot_downstream_influence(
    influences["WT"], title="Wild-type liver: metabolites predicted from the transcripts")
plt.show()

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/4107867202.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## The wiring

In [33]:
wiring = []
for genotype, graph in graphs.items():
    motifs = regulatory_motifs(graph, responsive_only=True)
    convergence = convergence_significance(graph, n_randomisations=200)
    vulnerability = structural_vulnerability(responsive_subnetwork(graph), top_n=5)
    wiring.append({
        "genotype": genotype,
        **{k: v for k, v in motifs["counts"].items()},
        "convergent reactions": convergence["observed"],
        "expected": round(convergence["null_mean"], 1),
        "z": round(convergence["z"], 1),
        "p": float(f"{convergence['p_value']:.3g}"),
        "holds it together": ", ".join(vulnerability["name"].head(2))
        if not vulnerability.empty else "",
    })
pd.DataFrame(wiring).set_index("genotype")

No responsive catalyst reaches any reaction: convergence is 0 by construction, not by measurement. This network has no measured Proteome layer, or none of its enzymes changed.


No responsive catalyst reaches any reaction: convergence is 0 by construction, not by measurement. This network has no measured Proteome layer, or none of its enzymes changed.


,product_inhibition,convergent reactions,expected,z,p,holds it together
genotype,,,,,,
WT,1,0,0.0,0.0,1.0,
ob/ob,2,0,0.0,0.0,1.0,


The convergence test counts reactions where a changed *protein* and a
changed metabolite meet. This study has no proteome, so the count is 0 by
construction, not because the layers fail to converge. The figure is shown
for completeness.

In [34]:
from transnet.visualization import (
    plot_convergence_null,
    plot_regulatory_motifs,
    plot_structural_vulnerability,
)

convergence_figure = plot_convergence_null(
    convergence_significance(graphs["WT"], n_randomisations=200),
    title="Wild-type liver: convergence against the shuffled null")
plt.show()

No responsive catalyst reaches any reaction: convergence is 0 by construction, not by measurement. This network has no measured Proteome layer, or none of its enzymes changed.


/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/413906276.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [35]:
motif_figure = plot_regulatory_motifs(
    regulatory_motifs(graphs["WT"], responsive_only=True),
    title="Wild-type liver: regulatory motifs")
plt.show()

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/4272441887.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The molecules whose removal would split the wild-type response into separate
pieces.

In [36]:
vulnerability_figure = plot_structural_vulnerability(
    structural_vulnerability(responsive_subnetwork(graphs["WT"]), top_n=10),
    title="Wild-type liver: molecules that hold the response together")
plt.show()

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/404585046.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## The two genotypes as networks

In [37]:
comparison = compare_transomic_networks(
    responsive_subnetwork(graphs["WT"]), responsive_subnetwork(graphs["ob/ob"]),
    "WT", "ob/ob",
)
print(f"edge Jaccard between the two responses: "
      f"{comparison['summary']['edge_jaccard']:.2f}")
comparison["edges_by_type"]

edge Jaccard between the two responses: 0.00


,edge_type,n_WT,n_ob/ob,n_shared,gained_in_ob/ob,lost_in_ob/ob
0,allosteric_activation,144,0,0,0,144
1,allosteric_inhibition,346,0,0,0,346
2,product,70,1,1,0,69
3,substrate,109,1,1,0,108


In [38]:
comparison["nodes_by_layer"]

,layer,n_WT,n_ob/ob,n_shared,unique_to_WT,unique_to_ob/ob
0,Metabolome,22,2,2,20,0
1,Reactions,259,1,1,258,0


In [39]:
from transnet.visualization import plot_condition_comparison

comparison_figure = plot_condition_comparison(comparison, "WT", "ob/ob")
plt.show()

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/3244079843.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figures and exports

Every figure is saved, and the wild-type responsive network is exported in
every format TransNet writes (see the *saving, exporting and sharing*
walkthrough).

In [41]:
from transnet.io import to_arena3d, to_cytoscape_json, to_transomics2cytoscape, write_network
from transnet.visualization import (
    plot_transomic_network,
    plot_transomic_network_interactive,
    transomic_backbone,
)

for figure, name in [
    (layer_figure, "layer_connectivity"),
    (timecourse_figure, "metabolites_over_time"),
    (similarity_figure, "gene_overlap"),
    (controversial_figures["WT"], "controversial_WT"),
    (controversial_figures["ob/ob"], "controversial_obob"),
    (axis_figures["WT"], "regulation_axes_WT"),
    (axis_figures["ob/ob"], "regulation_axes_obob"),
    (timing_figure, "temporal_structure"),
    (hub_figure, "transomic_hubs"),
    (influence_figure, "downstream_influence"),
    (vulnerability_figure, "structural_vulnerability"),
    (plot_transomic_network(responsive_subnetwork(graphs["WT"]),
                            title="Wild-type liver, 240 min after glucose"),
     "network_WT"),
    (plot_transomic_network(responsive_subnetwork(graphs["ob/ob"]),
                            title="ob/ob liver, 240 min after glucose"),
     "network_obob"),
    (plot_axis_composition(composition), "axes_by_genotype"),
    (tf_figure, "tf_activity"),
    (regulator_figure, "metabolite_regulators"),
    (comparison_figure, "genotypes_compared"),
    (convergence_figure, "convergence_null"),
    (motif_figure, "regulatory_motifs"),
    (saturation_figure, "saturation"),
    (paths_figure, "regulatory_paths"),
]:
    figure.savefig(OUT / f"{name}.png", dpi=150, bbox_inches="tight")
plt.show()

responsive = responsive_subnetwork(graphs["WT"])
exports = OUT / "exports"
exports.mkdir(exist_ok=True)
write_network(responsive, str(exports / "csv"))
to_cytoscape_json(responsive, exports / "network_cytoscape.json")
to_arena3d(responsive, exports / "network_arena3d.json")
to_transomics2cytoscape(responsive, zip_path=exports / "network_transomics2cytoscape.zip")
# a browser draws a few hundred nodes well; export the backbone the figures use
html_network = transomic_backbone(graphs["WT"], max_reactions=40)
plot_transomic_network_interactive(html_network, layout="layered",
                                   title="Wild-type liver, 240 min").write_html(
    exports / "network.html")
print(f"figures and tables in {OUT}")
print("exports:", ", ".join(sorted(p.name for p in exports.iterdir())))

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45850/2930299476.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


figures and tables in /Users/krv114/Documents/GitHub/transnet/data/published_results/liver_timecourse
exports: csv, network.html, network_arena3d.json, network_cytoscape.json, network_transomics2cytoscape.zip
